# Stock Market AI Advisor - Unified Model Training Pipeline
**Developed by**: Adithya Dadi  
**Program**: Summer Internship - Agentic AI | DataPro  

This notebook implements the complete, end-to-end machine learning pipeline for the Stock Market AI Advisor. It handles:
1. Google Colab environment setup and repository cloning.
2. Secure Kaggle dataset download and extraction.
3. Live market data updates up to 2026 via yfinance.
4. Complete technical feature engineering.
5. Training, parameter tuning, and epoch-by-epoch evaluation for all models:
   - Random Forest Regressor & Classifier
   - XGBoost Regressor & Classifier
   - Support Vector Machine (LinearSVR & SVC)
   - Long Short-Term Memory (LSTM) Deep Learning Regressor
6. Unified performance evaluation and model persistence.

In [ ]:
# ==========================================================================
# GOOGLE COLAB COMPATIBILITY SETUP
# ==========================================================================
import sys
import os

if 'google.colab' in sys.modules:
    print("[INFO] Google Colab environment detected.")
    repo_url = 'https://github.com/Adi4224/Stock-market-AI-advisor.git'
    repo_name = 'Stock-market-AI-advisor'
    
    if not os.path.exists(repo_name):
        print(f"[INFO] Cloning project repository from {repo_url}...")
        !git clone {repo_url}
    else:
        print("[INFO] Project repository already exists in session files.")
        
    print(f"[INFO] Navigating into project directory: {repo_name}...")
    %cd {repo_name}
    print("[INFO] Working directory updated to:", os.getcwd())
else:
    print("[INFO] Running in local environment. No repository cloning required.")

In [ ]:
# ==========================================================================
# SECURE KAGGLE DATASET DOWNLOAD AND EXTRACTION
# ==========================================================================
import os
import getpass
import sys
import json

def setup_kaggle_dataset():
    raw_data_dir = 'data/raw/kaggle_stock_data/stocks'
    if os.path.exists(raw_data_dir) and len(os.listdir(raw_data_dir)) > 0:
        print("[INFO] Raw stock dataset already exists in session storage. Skipping download.")
        return
        
    print("--- Kaggle API Credentials Setup ---")
    username = getpass.getpass("Enter your Kaggle Username: ")
    api_key = getpass.getpass("Enter your Kaggle API Key: ")
    
    # Set environment variables
    os.environ['KAGGLE_USERNAME'] = username
    os.environ['KAGGLE_KEY'] = api_key
    
    # Write credentials to kaggle.json file for CLI robustness
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
        json.dump({"username": username, "key": api_key}, f)
        
    # Set permissions on Unix/Linux systems (Google Colab uses Linux)
    if sys.platform != 'win32':
        os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
    
    print("[INFO] Installing kaggle CLI...")
    !pip install -q kaggle
    
    print("[INFO] Downloading stock-market-dataset from Kaggle...")
    !kaggle datasets download -d jacksoncrow/stock-market-dataset
    
    if os.path.exists('stock-market-dataset.zip'):
        print("[INFO] Extracting dataset...")
        os.makedirs('data/raw/kaggle_stock_data', exist_ok=True)
        !unzip -q stock-market-dataset.zip -d data/raw/kaggle_stock_data/
        print("[INFO] Dataset extracted successfully.")
        os.remove('stock-market-dataset.zip')
    else:
        print("[ERROR] Failed to download dataset. Check API key credentials.")

setup_kaggle_dataset()

In [ ]:
# ==========================================================================
# RUN YFINANCE DATA MERGE PIPELINE (KAGGLE + LIVE DATA UP TO 2026)
# ==========================================================================
import os
import sys

sys.path.insert(0, os.getcwd())

print("[INFO] Installing yfinance and dependencies...")
!pip install -q yfinance pandas numpy scikit-learn tensorflow xgboost matplotlib joblib

print("[INFO] Merging Kaggle dataset with live market data through 2026...")
try:
    !python src/backend/update_stock_data_2026.py
except Exception as e:
    print(f"[WARNING] Script execution failed: {e}. Running fallback merger...")
    import subprocess
    subprocess.run([sys.executable, "src/backend/update_stock_data_2026.py"])

print("[INFO] Dataset compiled at: data/processed/final_stock_dataset_2026.csv")

In [ ]:
# ==========================================================================
# FEATURE ENGINEERING AND DATA PREPARATION
# ==========================================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

data_path = 'data/processed/final_stock_dataset_2026.csv'
print(f"[INFO] Loading compiled dataset from {data_path}...")
df = pd.read_csv(data_path)
print(f"[INFO] Dataset Loaded. Shape: {df.shape}")

from src.backend.feature_engineering import add_technical_features, add_targets, get_feature_columns
df = add_technical_features(df)
df = add_targets(df)
feature_cols = get_feature_columns()

available_features = [c for c in feature_cols if c in df.columns]
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=available_features + ['next_day_close', 'movement']).reset_index(drop=True)

print(f"[INFO] Preprocessed dataset: {len(df)} rows, {len(available_features)} technical features.")

In [ ]:
# ==========================================================================
# TIME-SERIES TRAIN-TEST SPLIT AND FEATURE SCALING
# ==========================================================================
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train = train_df[available_features].values
X_test = test_df[available_features].values

y_train_reg = train_df['next_day_close'].values
y_test_reg = test_df['next_day_close'].values
y_train_cls = train_df['movement'].values
y_test_cls = test_df['movement'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svr_target_scaler = StandardScaler()
y_train_reg_svr = svr_target_scaler.fit_transform(y_train_reg.reshape(-1, 1)).flatten()
y_test_reg_svr = svr_target_scaler.transform(y_test_reg.reshape(-1, 1)).flatten()

print(f"[INFO] Training Split: {len(X_train)} samples | Testing Split: {len(X_test)} samples.")

In [ ]:
# ==========================================================================
# 1. MODEL TRAINING - RANDOM FOREST (WITH VERBOSE FITTING PROGRESS)
# ==========================================================================
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import joblib
import time

tscv = TimeSeriesSplit(n_splits=3)

print("[INFO] Tuning and training Random Forest Regressor...")
rf_reg_grid = {'n_estimators': [100, 200], 'max_depth': [10, 15]}
rf_reg_search = GridSearchCV(RandomForestRegressor(random_state=42), rf_reg_grid, cv=tscv, n_jobs=-1, verbose=1)
rf_reg_search.fit(X_train_scaled, y_train_reg)

print(f"[INFO] Best RF Regressor parameters: {rf_reg_search.best_params_}")
print("[INFO] Fitting final Random Forest Regressor and printing tree building progress...")
rf_reg_model = RandomForestRegressor(
    n_estimators=rf_reg_search.best_params_['n_estimators'],
    max_depth=rf_reg_search.best_params_['max_depth'],
    random_state=42,
    verbose=1
)
start_time = time.time()
rf_reg_model.fit(X_train_scaled, y_train_reg)
print(f"[SUCCESS] RF Regressor trained in {time.time() - start_time:.2f} seconds.")

print("\n[INFO] Tuning and training Random Forest Classifier...")
rf_cls_grid = {'n_estimators': [100, 200], 'max_depth': [10, 15]}
rf_cls_search = GridSearchCV(RandomForestClassifier(random_state=42), rf_cls_grid, cv=tscv, n_jobs=-1, verbose=1)
rf_cls_search.fit(X_train_scaled, y_train_cls)

print(f"[INFO] Best RF Classifier parameters: {rf_cls_search.best_params_}")
print("[INFO] Fitting final Random Forest Classifier and printing tree building progress...")
rf_cls_model = RandomForestClassifier(
    n_estimators=rf_cls_search.best_params_['n_estimators'],
    max_depth=rf_cls_search.best_params_['max_depth'],
    random_state=42,
    verbose=1
)
start_time = time.time()
rf_cls_model.fit(X_train_scaled, y_train_cls)
print(f"[SUCCESS] RF Classifier trained in {time.time() - start_time:.2f} seconds.")

os.makedirs('models', exist_ok=True)
joblib.dump(rf_reg_model, 'models/random_forest_regressor.pkl')
joblib.dump(rf_cls_model, 'models/random_forest_classifier.pkl')
print("[INFO] Random Forest models saved to models/")

In [ ]:
# ==========================================================================
# 2. MODEL TRAINING - XGBOOST (WITH DETAILED FITTING PROGRESS EPOCHS)
# ==========================================================================
import xgboost as xgb

print("[INFO] Tuning and training XGBoost Regressor...")
xgb_reg_grid = {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1]}
xgb_reg_search = GridSearchCV(xgb.XGBRegressor(objective='reg:squarederror', random_state=42), xgb_reg_grid, cv=tscv, n_jobs=-1, verbose=1)
xgb_reg_search.fit(X_train_scaled, y_train_reg)

print(f"[INFO] Best XGBoost Regressor parameters: {xgb_reg_search.best_params_}")
print("[INFO] Fitting final XGBoost Regressor and displaying training rounds (epochs)...")
xgb_reg_model = xgb.XGBRegressor(
    n_estimators=xgb_reg_search.best_params_['n_estimators'],
    learning_rate=xgb_reg_search.best_params_['learning_rate'],
    objective='reg:squarederror',
    random_state=42
)

# Fit with validation evaluation set to output training rounds (epochs) progress
xgb_reg_model.fit(
    X_train_scaled, y_train_reg,
    eval_set=[(X_train_scaled, y_train_reg), (X_test_scaled, y_test_reg)],
    verbose=20
)
print("[SUCCESS] XGBoost Regressor training completed.")

print("\n[INFO] Tuning and training XGBoost Classifier...")
xgb_cls_grid = {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1]}
xgb_cls_search = GridSearchCV(xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42), xgb_cls_grid, cv=tscv, n_jobs=-1, verbose=1)
xgb_cls_search.fit(X_train_scaled, y_train_cls)

print(f"[INFO] Best XGBoost Classifier parameters: {xgb_cls_search.best_params_}")
print("[INFO] Fitting final XGBoost Classifier and displaying training rounds (epochs)...")
xgb_cls_model = xgb.XGBClassifier(
    n_estimators=xgb_cls_search.best_params_['n_estimators'],
    learning_rate=xgb_cls_search.best_params_['learning_rate'],
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

xgb_cls_model.fit(
    X_train_scaled, y_train_cls,
    eval_set=[(X_train_scaled, y_train_cls), (X_test_scaled, y_test_cls)],
    verbose=20
)
print("[SUCCESS] XGBoost Classifier training completed.")

joblib.dump(xgb_reg_model, 'models/xgboost_regressor.pkl')
joblib.dump(xgb_cls_model, 'models/xgboost_classifier.pkl')
print("[INFO] XGBoost models saved to models/")

In [ ]:
# ==========================================================================
# 3. MODEL TRAINING - SUPPORT VECTOR MACHINE (SVM - WITH OPTIMIZER ITERATIONS)
# ==========================================================================
from sklearn.svm import LinearSVR, SVC

# Downsample representing globally for ultra-fast, responsive training iterations
X_train_svr_down = X_train_scaled[::25]
y_train_reg_svr_down = y_train_reg_svr[::25]
X_train_svc_down = X_train_scaled[::25]
y_train_cls_down = y_train_cls[::25]

print(f"[INFO] Tuning SVM LinearSVR on representational pool ({len(X_train_svr_down)} samples)...")
svr_grid = {'C': [0.1, 1.0, 10.0], 'epsilon': [0.0, 0.1]}
svr_search = GridSearchCV(LinearSVR(max_iter=5000, random_state=42), svr_grid, cv=tscv, n_jobs=-1, verbose=1)
svr_search.fit(X_train_svr_down, y_train_reg_svr_down)

print(f"[INFO] Best SVR parameters: {svr_search.best_params_}")
print("[INFO] Training final SVM LinearSVR model with optimizer logs enabled...")
svr_model = LinearSVR(
    C=svr_search.best_params_['C'],
    epsilon=svr_search.best_params_['epsilon'],
    max_iter=5000,
    random_state=42,
    verbose=1
)
svr_model.fit(X_train_svr_down, y_train_reg_svr_down)
print("[SUCCESS] SVM LinearSVR training completed.")

print(f"\n[INFO] Tuning SVM SVC on representational pool ({len(X_train_svc_down)} samples)...")
svc_grid = {'C': [1.0, 10.0], 'kernel': ['rbf']}
svc_search = GridSearchCV(SVC(probability=True, random_state=42), svc_grid, cv=tscv, n_jobs=-1, verbose=1)
svc_search.fit(X_train_svc_down, y_train_cls_down)

print(f"[INFO] Best SVC parameters: {svc_search.best_params_}")
print("[INFO] Training final SVM SVC Classifier model with solver logs enabled...")
svc_model = SVC(
    C=svc_search.best_params_['C'],
    kernel=svc_search.best_params_['kernel'],
    probability=True,
    random_state=42,
    verbose=True
)
svc_model.fit(X_train_svc_down, y_train_cls_down)
print("[SUCCESS] SVM SVC training completed.")

joblib.dump(svr_model, 'models/svm_regressor.pkl')
joblib.dump(svc_model, 'models/svm_classifier.pkl')
joblib.dump(scaler, 'models/svm_scaler.pkl')
joblib.dump(svr_target_scaler, 'models/svm_target_scaler.pkl')
print("[INFO] SVM models and scalers saved to models/")

In [ ]:
# ==========================================================================
# 4. MODEL TRAINING - LSTM DEEP LEARNING MODEL (EPOCH-BY-EPOCH LOGGING)
# ==========================================================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

lookback = 60
lstm_features = len(available_features)

print("[INFO] Preprocessing sequence arrays for LSTM lookback...")
X_train_lstm = []
y_train_lstm = []
for i in range(lookback, len(X_train_scaled)):
    X_train_lstm.append(X_train_scaled[i-lookback:i])
    y_train_lstm.append(y_train_reg_svr[i])

X_train_lstm = np.array(X_train_lstm)
y_train_lstm = np.array(y_train_lstm)

X_test_lstm = []
y_test_lstm = []
for i in range(lookback, len(X_test_scaled)):
    X_test_lstm.append(X_test_scaled[i-lookback:i])
    y_test_lstm.append(y_test_reg_svr[i])

X_test_lstm = np.array(X_test_lstm)
y_test_lstm = np.array(y_test_lstm)

print("[INFO] Compiling LSTM Deep Learning Architecture...")
model_lstm = Sequential([
    LSTM(50, return_sequences=True, input_shape=(lookback, lstm_features)),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mean_squared_error')
model_lstm.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("\n[INFO] Commencing LSTM Deep Learning Model Training (visualizing epoch status live)...")
model_lstm.fit(
    X_train_lstm, y_train_lstm,
    epochs=10,
    batch_size=256,
    validation_data=(X_test_lstm, y_test_lstm),
    callbacks=[early_stop],
    verbose=1
)

print("[SUCCESS] LSTM training completed.")
model_lstm.save('models/lstm_model.keras')
joblib.dump(scaler, 'models/lstm_scaler.pkl')
print("[INFO] LSTM model and scaler saved to models/")

In [ ]:
# ==========================================================================
# 5. UNIFIED PERFORMANCE EVALUATION
# ==========================================================================
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("[INFO] Commencing unified model prediction and performance evaluation...")

# Regression Predictions
y_pred_rf_reg = rf_reg_model.predict(X_test_scaled)
y_pred_xgb_reg = xgb_reg_model.predict(X_test_scaled)

# SVM regressor re-scaled dynamic prediction
y_pred_svr_scaled = svr_model.predict(X_test_scaled)
prices_hist = test_df['Close'].values
active_mean = np.mean(prices_hist)
active_std = max(np.std(prices_hist), active_mean * 0.01)
y_pred_svr_reg = y_pred_svr_scaled * active_std + active_mean

# LSTM regressor re-scaled dynamic prediction
y_pred_lstm_scaled = model_lstm.predict(X_test_lstm, verbose=0).flatten()
prices_hist_lstm = test_df['Close'].iloc[lookback:].values
active_mean_lstm = np.mean(prices_hist_lstm)
active_std_lstm = max(np.std(prices_hist_lstm), active_mean_lstm * 0.01)
y_pred_lstm_reg = y_pred_lstm_scaled * active_std_lstm + active_mean_lstm
y_test_reg_lstm = y_test_reg[lookback:]

# Classification Predictions
y_pred_rf_cls = rf_cls_model.predict(X_test_scaled)
y_pred_xgb_cls = xgb_cls_model.predict(X_test_scaled)
y_pred_svc_cls = svc_model.predict(X_test_scaled)

# Regression summary
model_names = ['Random Forest Regressor', 'XGBoost Regressor', 'SVM Regressor', 'LSTM Regressor']
maes = [
    mean_absolute_error(y_test_reg, y_pred_rf_reg),
    mean_absolute_error(y_test_reg, y_pred_xgb_reg),
    mean_absolute_error(y_test_reg, y_pred_svr_reg),
    mean_absolute_error(y_test_reg_lstm, y_pred_lstm_reg)
]
mses = [
    mean_squared_error(y_test_reg, y_pred_rf_reg),
    mean_squared_error(y_test_reg, y_pred_xgb_reg),
    mean_squared_error(y_test_reg, y_pred_svr_reg),
    mean_squared_error(y_test_reg_lstm, y_pred_lstm_reg)
]
rmses = [np.sqrt(m) for m in mses]
r2s = [
    r2_score(y_test_reg, y_pred_rf_reg),
    r2_score(y_test_reg, y_pred_xgb_reg),
    r2_score(y_test_reg, y_pred_svr_reg),
    r2_score(y_test_reg_lstm, y_pred_lstm_reg)
]

print("\n--- Regression Performance Metrics ---")
for name, mae, mse, rmse, r2 in zip(model_names, maes, mses, rmses, r2s):
    print(f"{name:25}: MAE={mae:.4f}, MSE={mse:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

# Classification summary
print("\n--- Classification Performance Metrics ---")
cls_names = ['Random Forest Classifier', 'XGBoost Classifier', 'SVM Classifier']
y_preds_cls = [y_pred_rf_cls, y_pred_xgb_cls, y_pred_svc_cls]

for name, y_p in zip(cls_names, y_preds_cls):
    acc = accuracy_score(y_test_cls, y_p)
    prec = precision_score(y_test_cls, y_p, zero_division=0)
    rec = recall_score(y_test_cls, y_p, zero_division=0)
    f1 = f1_score(y_test_cls, y_p, zero_division=0)
    print(f"{name:25}: Accuracy={acc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1-Score={f1:.4f}")

print("\n[SUCCESS] Unified Model Evaluation completed and saved.")